In [1]:
#Step 1: Install Pins & Load Model

import subprocess, sys

# Pins setup from shared scaffold
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

# CELL A: Transformers direct load (No vLLM to prevent numpy conflicts)
pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda"
)

installing: transformers==4.46.* accelerate==1.1.*


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [3]:
#Step 2: Measure TTFT and TPOT by Streaming

import time, threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**enc, max_new_tokens=new_tokens, do_sample=False, streamer=streamer)
    th = threading.Thread(target=model.generate, kwargs=kwargs)
    t0 = time.time()
    th.start()
    stamps = []
    for _ in streamer:
        stamps.append(time.time())
    th.join()
    ttft = stamps[0] - t0
    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0
    total = stamps[-1] - t0
    return {
        "ttft_s": round(ttft, 4),
        "tpot_s": round(tpot, 4),
        "total_s": round(total, 4),
        "n_tokens": len(stamps)
    }

# Mandatory CUDA warm-up generation (throwaway)
measure_stream(prompt_of_len(128), new_tokens=8)

ttft_by_len = {}
for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(n, r)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


128 {'ttft_s': 0.0433, 'tpot_s': 0.0338, 'total_s': 4.3746, 'n_tokens': 129}
512 {'ttft_s': 0.0662, 'tpot_s': 0.0531, 'total_s': 6.8655, 'n_tokens': 129}
2048 {'ttft_s': 0.3092, 'tpot_s': 0.0341, 'total_s': 4.6795, 'n_tokens': 129}


In [4]:
#Step 3: KV Cache Growth vs. Arithmetic

import gc, json

def kv_formula_kb_per_token(layers=28, kv_heads=2, head_dim=128, dbytes=2):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024  # 28.0 KB

def cache_bytes(pkv):
    if hasattr(pkv, "key_cache"):
        tensors = list(pkv.key_cache) + list(pkv.value_cache)
    else:
        tensors = [t for layer in pkv for t in layer]
    return sum(t.numel() * t.element_size() for t in tensors)

def measure_kv(context: int, new_tokens: int = 256):
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()
    enc = tok(prompt_of_len(context), return_tensors="pt").to("cuda")
    before = torch.cuda.memory_allocated()
    out = model.generate(**enc, max_new_tokens=new_tokens, do_sample=False,
                         use_cache=True, return_dict_in_generate=True)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    total_tokens = out.sequences.shape[1]
    return {
        "context": context,
        "total_tokens": int(total_tokens),
        "peak_kb_per_token": round((peak - before) / total_tokens / 1024, 1),
        "kv_kb_per_token": round(cache_bytes(out.past_key_values) / total_tokens / 1024, 1),
    }

formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)
kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]
for r in kv_rows:
    print(r, "  vs formula", formula, "KB/token")

with open("kv_check.json", "w") as f:
    json.dump({
        "formula_kb_per_token": formula,
        "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
        "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"]
    }, f)

formula KB/token: 28.0


From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 63.4, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 84.0, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 87.6, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token


In [5]:
#Step 4: Hand-Rolled Static Batching

QUEUE = [32, 32, 32, 256] * 6

def static_queue(batch: int, prompt: str = "Explain what an inference server does."):
    t0 = time.time(); useful = 0; slots = 0
    for i in range(0, len(QUEUE), batch):
        chunk = QUEUE[i:i + batch]
        n = max(chunk)
        enc = tok([prompt] * len(chunk), return_tensors="pt", padding=True).to("cuda")
        model.generate(**enc, max_new_tokens=n, do_sample=False)
        useful += sum(chunk)
        slots += n * len(chunk)
    dt = time.time() - t0
    return {
        "batch": batch,
        "wall_s": round(dt, 2),
        "tokens_per_s": round(useful / dt, 1),
        "slot_efficiency": round(useful / slots, 3)
    }

batch_rows = {}
for n in [1, 4, 8]:
    r = static_queue(n)
    batch_rows[str(n)] = r["tokens_per_s"]
    print(r)

{'batch': 1, 'wall_s': 64.01, 'tokens_per_s': 33.0, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 45.24, 'tokens_per_s': 46.7, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 22.79, 'tokens_per_s': 92.7, 'slot_efficiency': 0.344}


In [6]:
#Step 5: Export & Download Baseline Artifacts

import json
from google.colab import files

baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {k: v for k, v in batch_rows.items()},
}

with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)

print(json.dumps(baselines, indent=2))

# Mandatory download step for tomorrow's A/B testing baseline
files.download("baselines.json")

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.0433,
    "512": 0.0662,
    "2048": 0.3092
  },
  "tpot_s": 0.0343,
  "batch": {
    "1": 33.0,
    "4": 46.7,
    "8": 92.7
  }
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# Step 6: Verify

# Green-check verifier for Lab W3D1 (profile inference).
# Paste this as the last cell of your day-1 notebook and run it. It reads
# profile.json (the matrix rows you wrote) and checks the schema and the sanity
# rules. It also reads the batch experiment numbers if you saved them to
# batch_check.json; if that file is absent it asks for the two numbers inline so
# the batch-8 > batch-1 rule can still be checked.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os

REQUIRED_KEYS = {"dtype", "context", "vram_gb", "util_mean", "tokens_per_s"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def load_json(path: str):
    if not os.path.exists(path):
        fail(f"{path} not found; write it in the last data cell")
    try:
        with open(path) as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")


def main() -> None:
    rows = load_json("profile.json")

    if not isinstance(rows, list) or not rows:
        fail("profile.json must be a non-empty list of rows")

    # schema
    for i, row in enumerate(rows):
        if not isinstance(row, dict):
            fail(f"row {i} is not an object")
        missing = REQUIRED_KEYS - set(row)
        if missing:
            fail(f"row {i} missing keys: {sorted(missing)}")

    dtypes = {r["dtype"] for r in rows}
    contexts = sorted({r["context"] for r in rows})
    if "fp16" not in dtypes:
        fail("no fp16 rows; the matrix needs fp16")
    if len(contexts) < 3:
        fail(f"need at least 3 context lengths, found {contexts}")

    # sanity 1: VRAM rises with context (within each dtype)
    for dt in dtypes:
        sub = sorted((r for r in rows if r["dtype"] == dt),
                     key=lambda r: r["context"])
        vrams = [r["vram_gb"] for r in sub]
        if any(b < a - 0.01 for a, b in zip(vrams, vrams[1:])):
            fail(f"{dt} VRAM does not rise with context: {vrams}")

    # sanity 2: fp16 uses more memory than int8 at a shared context
    if "int8" in dtypes:
        shared = None
        for c in contexts:
            has_fp16 = any(r["dtype"] == "fp16" and r["context"] == c for r in rows)
            has_int8 = any(r["dtype"] == "int8" and r["context"] == c for r in rows)
            if has_fp16 and has_int8:
                shared = c
                break
        if shared is None:
            fail("fp16 and int8 share no context length to compare")
        fp16_v = next(r["vram_gb"] for r in rows
                      if r["dtype"] == "fp16" and r["context"] == shared)
        int8_v = next(r["vram_gb"] for r in rows
                      if r["dtype"] == "int8" and r["context"] == shared)
        if not fp16_v > int8_v:
            fail(f"fp16 VRAM ({fp16_v}) not above int8 VRAM ({int8_v}) at "
                 f"context {shared}")

    # sanity 3: batch-8 tokens/s beats batch-1
    b1 = b8 = None
    if os.path.exists("batch_check.json"):
        bc = load_json("batch_check.json")
        b1 = bc.get("batch1_tokens_per_s")
        b8 = bc.get("batch8_tokens_per_s")
    else:
        # allow the two numbers as module-level names set in an earlier cell
        b1 = globals().get("BATCH1_TOKENS_PER_S")
        b8 = globals().get("BATCH8_TOKENS_PER_S")
    if b1 is None or b8 is None:
        fail("batch numbers missing; save batch_check.json with "
             "batch1_tokens_per_s and batch8_tokens_per_s, or set "
             "BATCH1_TOKENS_PER_S / BATCH8_TOKENS_PER_S")
    if not b8 > b1:
        fail(f"batch-8 tokens/s ({b8}) not above batch-1 ({b1})")

    print(f"rows: {len(rows)}, dtypes: {sorted(dtypes)}, contexts: {contexts}")
    print(f"batch-1 tokens/s: {b1}, batch-8 tokens/s: {b8}")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 33.0, batch-8 tokens/s: 92.7
GREEN CHECK: PASS
